# Clean returns

Turns `returns_raw`, the returns API's returns as the *Copy returns* pipeline lands them, into `returns_clean`: one
row per return, with a reason we can count. The *Returns by reason* report reads `returns_clean` in Direct Lake mode,
so it's up to date as soon as this notebook finishes.

Scheduled every day at 06:30, after the pipeline. Default lakehouse: `contoso_sales`. Owner: Javier.

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

raw = spark.read.table("returns_raw")
orders = spark.read.table("orders").select("order_id", "delivered_at")
# TODO(javier): add orders_history once the shortcut is in (add_order_history_shortcut.py), so returns of orders
#  older than 12 months get their delivery date too.

## Reasons

The assistant sends one of four codes. The web form and the phone team send what the customer said, in English or
Spanish, so it's matched to the same codes by its words. The first reason with a match wins, so the telling ones go
first: "arrived broken and too small" is *damaged*. Anything else is *other*.

In [ ]:
REASONS = {
    "damaged": [
        "damaged", "broken", "torn", "ripped", "leak", "faulty", "dañad", "roto", "rota", "rasgad", "defectuos",
    ],
    "wrong_item": ["wrong (item|size|colou?r)", "not what i ordered", "too (small|big|large)", "equivocad", "talla"],
    "not_as_described": ["not as described", "different from", "looks different", "no es como", "distint"],
    "no_longer_needed": [
        "changed my mind", "no longer need", "don'?t need", "ya no (lo|la) necesito", "cambi[eé] de opini[oó]n",
    ],
}


def reason_code(reason):
    text = F.lower(F.trim(reason))
    code = F.when(text.isin(*REASONS), text)
    for name, words in REASONS.items():
        code = code.when(text.rlike("|".join(words)), name)
    return code.otherwise("other")

In [ ]:
latest = Window.partitionBy("rma").orderBy(F.col("updated_at").desc())

returns = (
    raw.filter(~F.col("order_id").startswith("TEST-"))  # the returns API's smoke tests, after each of its deploys
    .withColumn("n", F.row_number().over(latest))
    .filter("n = 1")  # the pipeline adds a row each day a return changes: the last one is where it stands
    .join(orders, "order_id", "left")
    .select(
        "rma",
        "order_id",
        F.to_date("opened_at").alias("opened_on"),
        "opened_by",  # assistant, agent or web
        reason_code(F.col("reason")).alias("reason"),
        "resolution",  # undecided, replacement or refund
        "status",
        F.datediff("opened_at", "delivered_at").alias("days_after_delivery"),
    )
)

returns.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("returns_clean")

## Check

The last 30 days by reason. Returns with no delivery date are for orders older than 12 months, which aren't in the
lakehouse until the order history is. If *other* is more than one return in ten, `REASONS` needs more words.

In [ ]:
recent = spark.read.table("returns_clean").filter(F.col("opened_on") >= F.date_sub(F.current_date(), 30))

display(
    recent.groupBy("reason")
    .agg(
        F.count("*").alias("returns"),
        F.round(F.avg("days_after_delivery"), 1).alias("days_after_delivery"),
        F.sum(F.col("days_after_delivery").isNull().cast("int")).alias("no_delivery_date"),
    )
    .orderBy(F.desc("returns"))
)